# Part D: Q1 revision experiments (items 1-4)

Run this notebook **after** `xai_comparison_harness_v5.ipynb` has been run end-to-end with
`PREPROCESSING = 'log2_loggenes'`. It reads that run's Drive checkpoints and data, and writes everything
new to `pancreatic_paper_checkpoints_log2_loggenes/partD/`. Every slow step checkpoints, so after a
disconnect just *Run all* again and finished steps are skipped.

| Cell | Item | What it does | CPU time (approx.) |
|---|---|---|---|
| D0-D1 | setup | installs, Drive, data, models, helpers; checks it reproduces the v5 C-indices | 3 min |
| D2 | 2. Cohort cleaning | GDC histology + ABSOLUTE purity; re-runs prediction and explanations on the clean PDAC subsets | 25 min |
| D3 | 3. Preprocessing grid | 2 transforms x 3 gene selections x 3 gene counts x 2 scalers x 2 seeds x 5 explainers | 90 min |
| D4 | 3. Variance decomposition | how much of the explanation disagreement each choice causes (Shapley R^2, one-factor-at-a-time) | 2 min |
| D5 | 4. Survival explainers | time-dependent IG, SurvSHAP(t), SurvLIME at 6/12/24 months; RSF contrast | 20 min |
| D6 | 4. Survival metrics | Harrell/Uno C, td-AUC 6/12/24, IBS, calibration, D-calibration, bootstrap CIs | 10 min |
| D7 | 1. External cohorts | downloads and harmonises GEO PDAC cohorts | 10 min |
| D8 | 1. External validation | transported C-index, explanation transfer, cohort-native models, gene-level replication | 20 min |
| D9 | all | summary JSON + LaTeX tables | seconds |

Set `QUICK_RUN = True` in D0 for a 10-minute dry run with tiny budgets (numbers are NOT for the paper).

In [ ]:
# ===== D0. Setup =====
import os, sys, subprocess, json, pickle, time, re, tarfile, urllib.request, warnings, copy
from itertools import combinations, product
from math import factorial

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lifelines', 'shap', 'lime', 'captum',
                    'scikit-survival', 'GEOparse'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

QUICK_RUN = os.environ.get('PAPER_QUICK_RUN') == '1'      # True = tiny budgets for a dry run only
BASE      = os.environ.get('PAPER_BASE', '/content/drive/MyDrive')
DATA_DIR  = f'{BASE}/pancreatic_paper_data'
V5_DIR    = f'{BASE}/pancreatic_paper_checkpoints_log2_loggenes'    # the v5 run this notebook builds on
OUT       = f'{V5_DIR}/partD' + ('_quick' if QUICK_RUN else '')
for d in (DATA_DIR, OUT, f'{OUT}/grid'):
    os.makedirs(d, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import RobustScaler, StandardScaler
from scipy.stats import spearmanr, rankdata, chi2, kruskal, binomtest
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lime.lime_tabular import LimeTabularExplainer
from captum.attr import IntegratedGradients
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc, integrated_brier_score
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
warnings.filterwarnings('ignore')
torch.set_num_threads(max(1, os.cpu_count() or 1))


def save(name, obj):
    path = f'{OUT}/{name}.pkl'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


def load(name):
    path = f'{OUT}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


def load_v5(name):
    path = f'{V5_DIR}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


RESULTS = load('results') or {}   # every headline number lands here; D9 writes it to JSON


def record(key, value):
    RESULTS[key] = value
    save('results', RESULTS)


METHOD_COLORS = {'IG': '#1baf7a', 'SHAP': '#2a78d6', 'LIME': '#eb6834', 'PFI': '#eda100', 'PDP': '#e87ba4'}
print(f"QUICK_RUN = {QUICK_RUN}; outputs -> {OUT}")

In [ ]:
# ===== D0b. Find or rebuild the v5 data file (tcga_paad_clean_log2.csv) =====
# v5 builds this file in its rebuild cell. If it is not on Drive (v5 not run with PREPROCESSING = 'log2_loggenes'
# on this account, or saved elsewhere), this cell (1) looks for it anywhere in My Drive, else (2) rebuilds it
# exactly as v5 does from the original tcga_paad_clean.csv (found on Drive or uploaded when asked) and cBioPortal.
import glob, shutil
DATA_CSV = f'{DATA_DIR}/tcga_paad_clean_log2.csv'


def _find(pattern):
    for p in [pattern, f'{DATA_DIR}/{pattern}', f'{BASE}/{pattern}']:
        if os.path.exists(p):
            return p
    hits = glob.glob(f'{BASE}/**/{pattern}', recursive=True)
    return sorted(hits, key=os.path.getmtime)[-1] if hits else None


if not os.path.exists(DATA_CSV):
    found = _find('tcga_paad_clean_log2.csv')
    if found:
        shutil.copy(found, DATA_CSV)
        print(f"Found the v5 data file at {found}; copied to {DATA_CSV}")
    else:
        old_path = _find('tcga_paad_clean.csv')
        if old_path is None and IN_COLAB:
            print("Upload the ORIGINAL tcga_paad_clean.csv (the 177-patient file used in v5):")
            from google.colab import files
            up = files.upload()
            old_path = next(k for k in up if k.endswith('.csv'))
        if old_path is None:
            raise FileNotFoundError("Neither tcga_paad_clean_log2.csv nor tcga_paad_clean.csv found.")
        print(f"Rebuilding the log2 data file from {old_path} (same steps as the v5 rebuild cell)...")
        STUDY = 'paad_tcga_pan_can_atlas_2018'
        rna_local = f'{DATA_DIR}/{STUDY}__data_mrna_seq_v2_rsem.txt'
        if not os.path.exists(rna_local):
            print("  downloading the TCGA-PAAD RNA file from the cBioPortal datahub (one-off, a few minutes)...")
            urllib.request.urlretrieve(f'https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/{STUDY}/data_mrna_seq_v2_rsem.txt', rna_local)
        old = pd.read_csv(old_path)
        id_col = next(c for c in old.columns[:5] if old[c].astype(str).str.startswith('TCGA-').all())
        lead_cols = old.columns[:5].tolist()
        rna = pd.read_csv(rna_local, sep='\t')
        rna = rna.dropna(subset=['Hugo_Symbol']).drop(columns=[c for c in ['Entrez_Gene_Id'] if c in rna.columns])
        rna = rna.groupby('Hugo_Symbol').mean()
        expr = rna.T
        expr = expr[expr.index.str[13:15] == '01']
        expr.index = expr.index.str[:12]
        expr = expr[~expr.index.duplicated(keep='first')]
        pid = old[id_col].astype(str).str[:12]
        missing = sorted(set(pid) - set(expr.index))
        if missing:
            raise RuntimeError(f"{len(missing)} patients not in the RNA file, e.g. {missing[:3]}")
        expr = expr.loc[pid.values].astype(float)
        logx = np.log2(expr.clip(lower=0) + 1)
        logx = logx.loc[:, logx.notna().all()]
        logx = logx.loc[:, (logx >= 1).mean() >= 0.20]
        top = logx.var().sort_values(ascending=False).index[:1000]
        new = pd.concat([old[lead_cols].reset_index(drop=True), logx[top].reset_index(drop=True)], axis=1)
        new.to_csv(DATA_CSV, index=False)
        print(f"  saved {DATA_CSV}, shape {new.shape}")
else:
    print(f"Found {DATA_CSV}")

if not os.path.exists(f'{V5_DIR}/cv_state.pkl'):
    _alt = glob.glob(f'{BASE}/**/pancreatic_paper_checkpoints_log2_loggenes*/cv_state.pkl', recursive=True)
    print("Note: v5 checkpoints not found in", V5_DIR,
          "-- D1 will retrain the final model with the v5 settings (same seed), and D6 will recompute the "
          "out-of-fold predictions. Results are equivalent but the v5 reproduction check is skipped."
          + (f" (Found checkpoints at {os.path.dirname(_alt[0])}: set V5_DIR to that folder in D0 and re-run.)" if _alt else ""))

In [ ]:
# ===== D1. Data, models and helpers (identical to v5 so the v5 numbers are reproduced) =====
df = pd.read_csv(DATA_CSV)
stage_map = {'STAGE I': 1, 'STAGE IA': 1, 'STAGE IB': 1, 'STAGE II': 2, 'STAGE IIA': 2, 'STAGE IIB': 2,
             'STAGE III': 3, 'STAGE IV': 4}
df['STAGE_ENCODED'] = df['AJCC_PATHOLOGIC_TUMOR_STAGE'].map(stage_map).fillna(2)
gene_cols = df.columns[5:1005].tolist()
CLIN_COLS = ['AGE', 'STAGE_ENCODED']
NC = len(CLIN_COLS)
ID_COL = next(c for c in df.columns[:5] if df[c].astype(str).str.startswith('TCGA-').all())
PID = df[ID_COL].astype(str).str[:12].values
N = len(df)
f32 = np.float32

Xc_raw = df[CLIN_COLS].values.astype(f32)
Xg_log = df[gene_cols].values.astype(f32)
if Xg_log.max() > 100:                       # guard: the rebuilt CSV already holds log2 values
    Xg_log = np.log2(Xg_log + 1)


def clip_pct(A, ref=None):
    ref = A if ref is None else ref
    return np.clip(A, np.percentile(ref, 1, axis=0), np.percentile(ref, 99, axis=0)).astype(f32)


Xg_clip = clip_pct(Xg_log)
T = df['OS_MONTHS'].values.astype(f32)
E = df['OS_STATUS'].values.astype(f32)
E_int = E.astype(int)
FEATURES = CLIN_COLS + gene_cols
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xc_raw, E))
TIMES = [6.0, 12.0, 24.0]
print(f"{N} patients, {len(gene_cols)} genes, {int(E.sum())} deaths, median follow-up {np.median(T):.1f} months")


def cox_ph_loss(risk, T_, E_):
    order = torch.argsort(T_, descending=True)
    risk, E_ = risk[order], E_[order]
    return -torch.mean((risk - torch.logcumsumexp(risk, dim=0)) * E_)


# Same layer order as v5, so the same torch seed gives the same initial weights.
class ClinicalMLP(nn.Module):
    kind = 'clin'

    def __init__(self, n_clin=2, n_gen=None):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                 nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, xc, xg=None):
        return self.net(xc).squeeze(1)


class GenomicMLP(nn.Module):
    kind = 'gen'

    def __init__(self, n_clin=None, n_gen=1000):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_gen, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
                                 nn.Linear(128, 1))

    def forward(self, xc, xg=None):
        return self.net(xg if xg is not None else xc).squeeze(1)


class GatedFusion(nn.Module):
    kind = 'fusion'

    def __init__(self, n_clin=2, n_gen=1000):
        super().__init__()
        self.clin_encoder = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                          nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gen_encoder = nn.Sequential(nn.Linear(n_gen, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                                         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4))
        self.gate_clin = nn.Sequential(nn.Linear(192, 64), nn.Sigmoid())
        self.gate_gen = nn.Sequential(nn.Linear(192, 128), nn.Sigmoid())
        self.head = nn.Linear(192, 1)

    def forward(self, xc, xg):
        hc, hg = self.clin_encoder(xc), self.gen_encoder(xg)
        h = torch.cat([hc, hg], 1)
        return self.head(torch.cat([hc * self.gate_clin(h), hg * self.gate_gen(h)], 1)).squeeze(1)


class ConcatFusion(nn.Module):
    kind = 'fusion'

    def __init__(self, n_clin=2, n_gen=1000):
        super().__init__()
        self.clin_encoder = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                          nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gen_encoder = nn.Sequential(nn.Linear(n_gen, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                                         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4))
        self.head = nn.Linear(192, 1)

    def forward(self, xc, xg):
        return self.head(torch.cat([self.clin_encoder(xc), self.gen_encoder(xg)], 1)).squeeze(1)


NETS = {'Clinical MLP': ClinicalMLP, 'Genomic MLP': GenomicMLP, 'Gated fusion': GatedFusion, 'Concat fusion': ConcatFusion}
EPOCHS = 30 if QUICK_RUN else 300


def _forward(model, xc, xg):
    if model.kind == 'clin':
        return model(xc)
    if model.kind == 'gen':
        return model(xg)
    return model(xc, xg)


def train_model(model, Xc, Xg, T_, E_, epochs=None, lr=1e-3):
    epochs = epochs or EPOCHS
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xc, Xg = torch.tensor(Xc, dtype=torch.float32), torch.tensor(Xg, dtype=torch.float32)
    Tt, Et = torch.tensor(T_, dtype=torch.float32), torch.tensor(E_, dtype=torch.float32)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = cox_ph_loss(_forward(model, Xc, Xg), Tt, Et)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
    return model.eval()


def risk_of(model, Xc, Xg):
    with torch.no_grad():
        return _forward(model, torch.tensor(Xc, dtype=torch.float32), torch.tensor(Xg, dtype=torch.float32)).numpy()


class Flat(nn.Module):
    """Wraps a two-input model as f([clinical | genes]) for the explainers; optional output transform."""
    def __init__(self, base, nc=NC, out=None):
        super().__init__()
        self.base, self.nc, self.out = base, nc, out

    def forward(self, x):
        r = _forward(self.base, x[:, :self.nc], x[:, self.nc:])
        return r if self.out is None else self.out(r)


def np_fn(module):
    def fn(X):
        with torch.no_grad():
            return module(torch.tensor(np.asarray(X, dtype=f32))).numpy()
    return fn


def harrell_c_many(T_, E_, R):
    """Harrell's C for every row of R; same pairing and tie rules as lifelines.concordance_index(T, -risk, E)."""
    T_ = np.asarray(T_, float); E_ = np.asarray(E_).astype(bool); R = np.atleast_2d(np.asarray(R, float))
    comp = ((T_[:, None] < T_[None, :]) | ((T_[:, None] == T_[None, :]) & ~E_[None, :])) & E_[:, None]
    n = comp.sum()
    if n == 0:
        return np.full(len(R), np.nan)
    gt = (R[:, :, None] > R[:, None, :]) & comp
    eq = (R[:, :, None] == R[:, None, :]) & comp
    return (gt.sum((1, 2)) + 0.5 * eq.sum((1, 2))) / n


def harrell_c(T_, E_, r):
    return float(harrell_c_many(T_, E_, np.asarray(r)[None])[0])


def ig_attr(module, X, baseline_vec, n_steps=50, target=None):
    X_t = torch.tensor(np.asarray(X, dtype=f32), requires_grad=True)
    base = torch.tensor(np.tile(np.asarray(baseline_vec, dtype=f32), (len(X), 1)))
    a = IntegratedGradients(module).attribute(X_t, baselines=base, n_steps=n_steps, target=target,
                                              internal_batch_size=4096)
    return a.detach().numpy()


def kernel_shap(fn, X, bg_pool, nsamples, seed=42, l1='num_features(50)', n_bg=30):
    """KernelSHAP as in v5 (background = shap.sample(pool, 30), which always uses random_state 0;
    `seed` only changes the coalition sampling). Returns (n, p) or (n, p, outputs)."""
    np.random.seed(seed)
    ex = shap.KernelExplainer(fn, shap.sample(bg_pool, min(n_bg, len(bg_pool))))
    sv = ex.shap_values(np.asarray(X, dtype=f32), nsamples=nsamples, l1_reg=l1, silent=True)
    if isinstance(sv, list):
        sv = np.stack([np.asarray(s) for s in sv], axis=-1)
    sv = np.asarray(sv)
    return sv.reshape(len(X), -1) if sv.ndim == 2 or (sv.ndim == 3 and sv.shape[-1] == 1) else sv


def lime_local(fn, X_explain, X_train, num_samples=300, seed=42):
    ex = LimeTabularExplainer(training_data=X_train, mode='regression', discretize_continuous=False, random_state=seed)
    out = np.zeros(X_explain.shape, dtype=float)
    for i in range(len(X_explain)):
        e = ex.explain_instance(X_explain[i], fn, num_features=X_explain.shape[1], num_samples=num_samples)
        for j, w in e.as_map()[1]:
            out[i, j] = w
    return out


def randomise(model, seed=0):
    torch.manual_seed(seed)
    rm = copy.deepcopy(model)
    for mod in rm.modules():
        if isinstance(mod, nn.Linear):
            mod.reset_parameters()
    return rm.eval()


def exact_group_shapley(fn, X, groups, background):
    names = list(groups); k = len(names); n = len(X)

    def v(S):
        out = np.zeros(n)
        for brow in background:
            Xs = np.tile(brow, (n, 1)).astype(f32)
            for g in S:
                Xs[:, groups[g]] = X[:, groups[g]]
            out += fn(Xs)
        return out / len(background)

    vals = {S: v(S) for r in range(k + 1) for S in combinations(names, r)}
    phi = {}
    for g in names:
        others = [h for h in names if h != g]
        tot = np.zeros(n)
        for r in range(k):
            for S in combinations(others, r):
                w = factorial(r) * factorial(k - r - 1) / factorial(k)
                tot += w * (vals[tuple(sorted(S + (g,), key=names.index))] - vals[S])
        phi[g] = tot
    return pd.DataFrame(phi)


def modality_share(fn, X, nc=NC, n_bg=60):
    bg = X[np.random.RandomState(0).choice(len(X), min(n_bg, len(X)), replace=False)]
    phi = exact_group_shapley(fn, X, {'clinical': list(range(nc)), 'genomic': list(range(nc, X.shape[1]))}, bg).abs()
    return float((phi['genomic'] / (phi.sum(1) + 1e-12)).mean())


def top_genes(g, names, k=20, nc=NC):
    o = np.argsort(-np.asarray(g))
    return [names[i] for i in o if i >= nc][:k]


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(1, len(a | b))


def breslow(r_tr, T_tr, E_tr):
    c = float(np.max(r_tr)); er = np.exp(np.asarray(r_tr) - c)
    ev = np.unique(T_tr[E_tr == 1])
    dH = np.array([((T_tr == t) & (E_tr == 1)).sum() / er[T_tr >= t].sum() for t in ev])
    return ev, np.cumsum(dH), c


def H0_at(bres, times):
    ev, H, _ = bres
    idx = np.searchsorted(ev, np.asarray(times, float), side='right') - 1
    return np.where(idx >= 0, H[np.clip(idx, 0, None)], 0.0)


def surv_from_risk(bres, r, times):
    return np.exp(-np.outer(np.exp(np.asarray(r) - bres[2]), H0_at(bres, times)))


def fit_coxnet(X, y, seed=0):
    path = CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.05, n_alphas=25, max_iter=100000).fit(X, y)
    gcv = GridSearchCV(CoxnetSurvivalAnalysis(l1_ratio=0.5, max_iter=100000), {'alphas': [[a] for a in path.alphas_]},
                       cv=KFold(3, shuffle=True, random_state=seed), error_score=0.5, n_jobs=1).fit(X, y)
    best = gcv.best_estimator_
    if np.count_nonzero(best.coef_) < 10:
        nz = (path.coef_ != 0).sum(0); j = int(np.argmin(np.abs(nz - 30)))
        best = CoxnetSurvivalAnalysis(l1_ratio=0.5, alphas=[path.alphas_[j]], max_iter=100000).fit(X, y)
    return best


def fit_rsf(X, y, n_estimators=300):
    return RandomSurvivalForest(n_estimators=30 if QUICK_RUN else n_estimators, min_samples_leaf=10,
                                max_features='sqrt', random_state=42, n_jobs=-1).fit(X, y)


def run_cv(Xc, Xg, T_, E_, models=('Clinical MLP', 'Genomic MLP', 'Gated fusion', 'Concat fusion'), folds=None, seed=42):
    """Out-of-fold risks; scalers fit on the training fold only (as in v5)."""
    folds = folds if folds is not None else list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xc, E_))
    out = {m: {'tr': [], 'te': []} for m in models}
    for tr, te in folds:
        sc, sg = RobustScaler(), RobustScaler()
        Xc_tr = sc.fit_transform(Xc[tr]).astype(f32); Xc_te = sc.transform(Xc[te]).astype(f32)
        Xg_tr = sg.fit_transform(Xg[tr]).astype(f32); Xg_te = sg.transform(Xg[te]).astype(f32)
        for m in models:
            if m in ('Coxnet', 'RSF'):
                A_tr, A_te = np.hstack([Xc_tr, Xg_tr]).astype(float), np.hstack([Xc_te, Xg_te]).astype(float)
                y = Surv.from_arrays(E_[tr].astype(bool), T_[tr])
                est = fit_coxnet(A_tr, y) if m == 'Coxnet' else fit_rsf(A_tr, y)
                out[m]['tr'].append(est.predict(A_tr)); out[m]['te'].append(est.predict(A_te))
            else:
                torch.manual_seed(seed)
                net = train_model(NETS[m](Xc.shape[1], Xg.shape[1]), Xc_tr, Xg_tr, T_[tr], E_[tr])
                out[m]['tr'].append(risk_of(net, Xc_tr, Xg_tr)); out[m]['te'].append(risk_of(net, Xc_te, Xg_te))
    return out, folds


def fit_full(Xc, Xg, T_, E_, seed=42, cls=GatedFusion):
    sc, sg = RobustScaler(), RobustScaler()
    A_c, A_g = sc.fit_transform(Xc).astype(f32), sg.fit_transform(Xg).astype(f32)
    torch.manual_seed(seed)
    m = train_model(cls(Xc.shape[1], Xg.shape[1]), A_c, A_g, T_, E_)
    return m, sc, sg, np.hstack([A_c, A_g]).astype(f32)


# --- Reproduction check against the v5 run (same data, seeds and folds -> same fold C-indices)
_v5cv = load_v5('cv_state')
if _v5cv is not None and not QUICK_RUN:
    chk = load('repro_check')
    if chk is None:
        oof, _ = run_cv(Xc_raw, Xg_clip, T, E, models=('Gated fusion',), folds=FOLDS)
        chk = [harrell_c(T[te], E_int[te], oof['Gated fusion']['te'][f]) for f, (tr, te) in enumerate(FOLDS)]
        save('repro_check', chk)
    diff = np.max(np.abs(np.array(chk) - np.array(_v5cv['results_cv']['fusion'])))
    print(f"Reproduction of v5 gated-fusion fold C-indices: max |diff| = {diff:.2e} "
          f"({'OK' if diff < 1e-3 else 'MISMATCH - check that v5 was run with log2_loggenes'})")

# The paper's final model (v5 cell 10), loaded from Drive so every explanation below refers to the same model
_ms = load_v5('model_full_state')
if _ms is not None:
    model_full = GatedFusion(); model_full.load_state_dict(_ms['model_full_state_dict']); model_full.eval()
    X_all = _ms['X_all'].astype(f32)
    print("Loaded the v5 final model (model_full_state).")
else:
    model_full, _, _, X_all = fit_full(Xc_raw, Xg_clip, T, E)
    print("v5 model checkpoint not found - retrained the final model with the v5 settings.")
F_full = Flat(model_full).eval()
pred_full = np_fn(F_full)
r_all = pred_full(X_all)
MED_ALL = np.median(X_all, axis=0).astype(f32)
IG_ALL = ig_attr(F_full, X_all, MED_ALL)
IG_GLOBAL = np.abs(IG_ALL).mean(0)
print(f"Final model in-sample C = {harrell_c(T, E_int, r_all):.4f}; IG top-5: {top_genes(IG_GLOBAL, FEATURES, 5)}")

In [ ]:
# ===== D2. Item 2 -- Cohort cleaning: histology and tumour purity =====
# Histology: GDC primary_diagnosis for every TCGA-PAAD case (public API, no login).
# Purity: ABSOLUTE purity from the TCGA PanCanAtlas (public GDC file); pathologist % tumour nuclei as a fallback.
import requests

PURITY_MIN = 0.20          # ABSOLUTE purity threshold for the "high-purity" subset; 0.30 is reported as a sensitivity
ABSOLUTE_URL = 'https://api.gdc.cancer.gov/data/4f277128-f793-4354-a13d-30cc7fe9f6b5'   # TCGA_mastercalls.abs_tables_JSedit.fixed.txt


def gdc_clinical():
    ck = load('gdc_clinical')
    if ck is not None:
        return ck
    filt = {"op": "in", "content": {"field": "project.project_id", "value": ["TCGA-PAAD"]}}
    fields = ','.join(['submitter_id', 'diagnoses.primary_diagnosis', 'diagnoses.morphology',
                       'samples.sample_type', 'samples.portions.slides.percent_tumor_nuclei'])
    r = requests.get('https://api.gdc.cancer.gov/cases', timeout=120,
                     params={'filters': json.dumps(filt), 'fields': fields, 'format': 'json', 'size': '1000'})
    r.raise_for_status()
    rows = []
    for h in r.json()['data']['hits']:
        dx = [d for d in (h.get('diagnoses') or []) if d.get('primary_diagnosis')] or [{}]
        nuclei = [float(sl['percent_tumor_nuclei'])
                  for s in (h.get('samples') or []) if s.get('sample_type') == 'Primary Tumor'
                  for p in (s.get('portions') or []) for sl in (p.get('slides') or [])
                  if sl.get('percent_tumor_nuclei') is not None]
        rows.append({'PATIENT_ID': h['submitter_id'], 'primary_diagnosis': dx[0].get('primary_diagnosis'),
                     'morphology': dx[0].get('morphology'), 'pct_tumor_nuclei': np.mean(nuclei) if nuclei else np.nan})
    out = pd.DataFrame(rows)
    save('gdc_clinical', out)
    return out


def absolute_purity():
    path = f'{DATA_DIR}/TCGA_mastercalls.abs_tables_JSedit.fixed.txt'
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(ABSOLUTE_URL, path)
        except Exception as e:
            print(f"  ABSOLUTE purity file not downloaded ({e}); falling back to pathology % tumour nuclei.")
            return None
    a = pd.read_csv(path, sep='\t')
    scol = next((c for c in a.columns if c.lower() in ('array', 'sample', 'sample_id', 'tcga_id')), a.columns[0])
    pcol = next(c for c in a.columns if c.lower() == 'purity')
    a = a[[scol, pcol]].dropna()
    a = a[a[scol].astype(str).str[13:15] == '01']
    a['PATIENT_ID'] = a[scol].astype(str).str[:12]
    return a.groupby('PATIENT_ID')[pcol].mean().rename('absolute_purity')


def histology_group(s):
    s = str(s).lower()
    if any(w in s for w in ('neuroendocrine', 'carcinoid', 'islet')):
        return 'neuroendocrine'
    if 'mucinous' in s or 'colloid' in s:
        return 'colloid/mucinous'
    if 'undifferentiated' in s or 'anaplastic' in s:
        return 'undifferentiated'
    if 'acinar' in s:
        return 'acinar'
    if 'duct' in s:
        return 'ductal'
    if 'adenocarcinoma' in s:
        return 'adenocarcinoma NOS'
    return 'other/unknown'


clin = gdc_clinical().drop_duplicates('PATIENT_ID').set_index('PATIENT_ID').reindex(PID)
clin['histology'] = clin['primary_diagnosis'].map(histology_group)
pur = absolute_purity()
clin['absolute_purity'] = pur.reindex(PID).values if pur is not None else np.nan
print("GDC primary diagnosis for the 177 patients:")
print(clin['primary_diagnosis'].fillna('missing').value_counts().to_string())
print("\nGrouped:"); print(clin['histology'].value_counts().to_string())
print(f"\nABSOLUTE purity available for {clin['absolute_purity'].notna().sum()}/{N}; "
      f"median {clin['absolute_purity'].median():.2f}; < {PURITY_MIN}: {(clin['absolute_purity'] < PURITY_MIN).sum()}")
print(f"Pathology % tumour nuclei available for {clin['pct_tumor_nuclei'].notna().sum()}/{N}; "
      f"median {clin['pct_tumor_nuclei'].median():.0f}%")

is_pdac = clin['histology'].isin(['ductal', 'adenocarcinoma NOS']).values
purity = clin['absolute_purity'].values.astype(float)
purity_src = 'ABSOLUTE'
if np.isnan(purity).all():
    purity, purity_src = clin['pct_tumor_nuclei'].values.astype(float) / 100, 'pathology % tumour nuclei'
COHORTS = {'All (paper)': np.ones(N, bool),
           'PDAC histology': is_pdac,
           f'PDAC + purity >= {PURITY_MIN:.2f}': is_pdac & (purity >= PURITY_MIN),
           'PDAC + purity >= 0.30': is_pdac & (purity >= 0.30)}
print(f"\nPurity source: {purity_src}")
for k, m in COHORTS.items():
    print(f"  {k:28s} n = {m.sum():3d}, deaths = {int(E[m].sum())}")

SHAP_BUDGET_D2 = 256 if QUICK_RUN else 2048


def cohort_analysis(name, mask):
    ck = load(f'cohort_{name}')
    if ck is not None:
        return ck
    t0 = time.time(); idx = np.where(mask)[0]
    Xc, Xg, Tc, Ec = Xc_raw[idx], Xg_clip[idx], T[idx], E[idx]
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xc, Ec))
    oof, _ = run_cv(Xc, Xg, Tc, Ec, models=('Clinical MLP', 'Genomic MLP', 'Gated fusion', 'Concat fusion', 'Coxnet'),
                    folds=folds)
    cv = {m: [harrell_c(Tc[te], Ec[te].astype(int), oof[m]['te'][f]) for f, (tr, te) in enumerate(folds)] for m in oof}
    model, sc, sg, X = fit_full(Xc, Xg, Tc, Ec)
    F = Flat(model).eval(); fn = np_fn(F); med = np.median(X, 0)
    ig = np.abs(ig_attr(F, X, med)).mean(0)
    sh = np.abs(kernel_shap(fn, X, X, SHAP_BUDGET_D2, seed=42, l1=False)).mean(0)
    ig_rand = np.abs(ig_attr(Flat(randomise(model)).eval(), X, med)).mean(0)
    res = {'n': int(mask.sum()), 'deaths': int(Ec.sum()), 'cv': cv, 'ig': ig, 'shap': sh,
           'sanity_top10_overlap': len(set(np.argsort(-ig)[:10]) & set(np.argsort(-ig_rand)[:10])) / 10,
           'genomic_share_exact': modality_share(fn, X), 'runtime_s': time.time() - t0}
    save(f'cohort_{name}', res)
    return res


rows, CO = [], {}
for name, mask in COHORTS.items():
    if mask.sum() < 40:
        print(f"  {name}: only {mask.sum()} patients - skipped"); continue
    CO[name] = res = cohort_analysis(name, mask)
    ref = CO['All (paper)']
    rows.append({'cohort': name, 'n': res['n'], 'deaths': res['deaths'],
                 **{f'C {m}': f"{np.mean(v):.3f} +/- {np.std(v):.3f}" for m, v in res['cv'].items()},
                 'IG sanity top-10 overlap': res['sanity_top10_overlap'],
                 'exact genomic share': round(res['genomic_share_exact'], 3),
                 'IG rho vs All': round(spearmanr(res['ig'], ref['ig']).correlation, 3),
                 'IG top-20 Jaccard vs All': round(jaccard(top_genes(res['ig'], FEATURES), top_genes(ref['ig'], FEATURES)), 3),
                 'SHAP-2048 vs IG rho': round(spearmanr(res['shap'], res['ig']).correlation, 3),
                 'IG top-5': ', '.join(top_genes(res['ig'], FEATURES, 5))})
D2 = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 60)
print(D2.to_string(index=False))
D2.to_csv(f'{OUT}/D2_cohort_cleaning.csv', index=False)

# Does the top gene reflect histology? Per-patient |IG| of the top genes by histology group (paper's final model)
TOPG = top_genes(IG_GLOBAL, FEATURES, 5)
hrows = []
for g in TOPG:
    j = FEATURES.index(g); a = np.abs(IG_ALL[:, j]); x = Xg_log[:, gene_cols.index(g)]
    grp = clin['histology'].values
    groups = [a[grp == h] for h in np.unique(grp) if (grp == h).sum() >= 3]
    try:
        p = kruskal(*groups).pvalue if len(groups) > 1 else np.nan
    except ValueError:
        p = np.nan
    for h in pd.Series(grp).value_counts().index:
        hrows.append({'gene': g, 'histology': h, 'n': int((grp == h).sum()),
                      'mean |IG|': a[grp == h].mean(), 'mean log2 expr': x[grp == h].mean(), 'Kruskal p (|IG|)': p})
D2h = pd.DataFrame(hrows)
print("\nTop IG genes by histology group (is the attribution concentrated in non-PDAC tumours?):")
print(D2h.round(3).to_string(index=False))
D2h.to_csv(f'{OUT}/D2_top_genes_by_histology.csv', index=False)
clin.reset_index().to_csv(f'{OUT}/D2_patient_histology_purity.csv', index=False)
record('D2_cohort_cleaning', {'table': D2.to_dict('records'), 'purity_source': purity_src,
                              'histology_counts': clin['histology'].value_counts().to_dict()})

In [ ]:
# ===== D3. Item 3 -- Full preprocessing grid =====
# Every combination of: value transform (raw RSEM / log2), gene selection (variance / MAD / univariate Cox score),
# number of genes (500 / 1,000 / 2,000), scaler (robust / standard) and model seed (0 / 1). For every pipeline
# the gated fusion model is retrained and explained by IG, KernelSHAP-200 (two independent runs),
# KernelSHAP-2048 (seed 0 only) and LIME-300, on a fixed set of patients. Selection is computed on the
# transformed scale; the CV C-index re-selects genes inside every training fold, so it has no leakage.
STUDY = 'paad_tcga_pan_can_atlas_2018'


def cbio_file(member):
    local = f'{DATA_DIR}/{STUDY}__{member}'
    if os.path.exists(local):
        return local
    print(f"Downloading {member} from the cBioPortal datahub (one-off)...")
    urllib.request.urlretrieve(f'https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/{STUDY}/{member}', local)
    return local


def full_expression():
    ck = load('full_expression')
    if ck is not None:
        return ck
    rna = pd.read_csv(cbio_file('data_mrna_seq_v2_rsem.txt'), sep='\t')
    rna = rna.dropna(subset=['Hugo_Symbol']).drop(columns=[c for c in ['Entrez_Gene_Id'] if c in rna.columns])
    rna = rna.groupby('Hugo_Symbol').mean()
    ex = rna.T
    ex = ex[ex.index.str[13:15] == '01']
    ex.index = ex.index.str[:12]
    ex = ex[~ex.index.duplicated(keep='first')].loc[PID].astype(f32)
    ex = ex.loc[:, ex.notna().all()].clip(lower=0)
    logx = np.log2(ex + 1)
    keep = (logx >= 1).mean() >= 0.20                 # same expression filter as the v5 rebuild cell
    out = {'raw': ex.loc[:, keep], 'log2': logx.loc[:, keep]}
    save('full_expression', out)
    return out


def cox_score_z(X, T_, E_):
    """Univariate Cox score-test z for every column at once (Breslow ties)."""
    X = np.asarray(X, float)
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-8)
    order = np.argsort(-np.asarray(T_), kind='stable')
    Xo, To, Eo = Xs[order], np.asarray(T_)[order], np.asarray(E_)[order]
    S1, S2 = np.cumsum(Xo, 0), np.cumsum(Xo ** 2, 0)
    last = np.searchsorted(-To, -To, side='right') - 1           # risk set = everyone with T >= T_i
    n_at = (last + 1)[:, None]
    m1, m2 = S1[last] / n_at, S2[last] / n_at
    ev = Eo == 1
    U = (Xo[ev] - m1[ev]).sum(0)
    I = (m2[ev] - m1[ev] ** 2).sum(0)
    return U / np.sqrt(I + 1e-12)


def select_genes(M, method, k, T_, E_):
    if method == 'variance':
        s = M.var()
    elif method == 'mad':
        s = (M - M.median()).abs().median() + 1e-9 * M.var()          # tiny variance term breaks MAD ties at 0
    else:
        s = pd.Series(np.abs(cox_score_z(M.values, T_, E_)), index=M.columns)
    return s.sort_values(ascending=False).index[:k].tolist()


EXPR = full_expression()
GRID = {'transform': ['raw', 'log2'], 'selection': ['variance', 'mad', 'cox'], 'k': [500, 1000, 2000],
        'scaler': ['robust', 'standard']}
GRID_SEEDS = [0, 1]
if QUICK_RUN:
    GRID = {'transform': ['raw', 'log2'], 'selection': ['variance', 'cox'], 'k': [200], 'scaler': ['robust']}
EXPLAIN_N, BG_N = (12, 8) if QUICK_RUN else (40, 20)
SHAP_LO, SHAP_HI, LIME_N = (64, 256, 100) if QUICK_RUN else (200, 2048, 300)
EXPLAIN_IDX = np.sort(np.random.RandomState(42).choice(N, EXPLAIN_N, replace=False))
PIPELINES = [dict(zip(GRID, v)) for v in product(*GRID.values())]
SCALERS = {'robust': RobustScaler, 'standard': StandardScaler}
print(f"{len(PIPELINES)} pipelines x {len(GRID_SEEDS)} seeds; explaining {EXPLAIN_N} fixed patients")


def pipe_key(p):
    return f"{p['transform']}_{p['selection']}_k{p['k']}_{p['scaler']}"


def run_pipeline(p):
    key = pipe_key(p); ck = load(f'grid/{key}')
    if ck is not None:
        return ck
    t0 = time.time()
    M = EXPR[p['transform']]
    Scaler = SCALERS[p['scaler']]
    # (a) honest CV C-index: gene selection, clipping and scaling are refit inside each training fold
    cv_c = []
    for tr, te in FOLDS:
        g_tr = select_genes(M.iloc[tr], p['selection'], p['k'], T[tr], E[tr])
        A = M[g_tr].values.astype(f32); A = clip_pct(A, ref=A[tr])
        sc, sg = Scaler(), Scaler()
        Xc_tr, Xc_te = sc.fit_transform(Xc_raw[tr]).astype(f32), sc.transform(Xc_raw[te]).astype(f32)
        Xg_tr, Xg_te = sg.fit_transform(A[tr]).astype(f32), sg.transform(A[te]).astype(f32)
        torch.manual_seed(0)
        m = train_model(GatedFusion(NC, len(g_tr)), Xc_tr, Xg_tr, T[tr], E[tr])
        cv_c.append(harrell_c(T[te], E_int[te], risk_of(m, Xc_te, Xg_te)))
    # (b) full-data models and their explanations
    genes = select_genes(M, p['selection'], p['k'], T, E)
    A = clip_pct(M[genes].values.astype(f32))
    sc, sg = Scaler(), Scaler()
    X = np.hstack([sc.fit_transform(Xc_raw), sg.fit_transform(A)]).astype(f32)
    Xe, med = X[EXPLAIN_IDX], np.median(X, 0)
    expl, sanity = {}, None
    for s in GRID_SEEDS:
        torch.manual_seed(s)
        model = train_model(GatedFusion(NC, len(genes)), X[:, :NC], X[:, NC:], T, E)
        F = Flat(model).eval(); fn = np_fn(F)
        expl[(s, 'IG', 0, 'a')] = np.abs(ig_attr(F, Xe, med)).mean(0)
        expl[(s, 'SHAP', SHAP_LO, 'a')] = np.abs(kernel_shap(fn, Xe, X, SHAP_LO, seed=100 + s, n_bg=BG_N)).mean(0)
        expl[(s, 'SHAP', SHAP_LO, 'b')] = np.abs(kernel_shap(fn, Xe, X, SHAP_LO, seed=200 + s, n_bg=BG_N)).mean(0)
        if s == 0:
            expl[(s, 'SHAP', SHAP_HI, 'a')] = np.abs(kernel_shap(fn, Xe, X, SHAP_HI, seed=300, l1=False, n_bg=BG_N)).mean(0)
            ig_r = np.abs(ig_attr(Flat(randomise(model)).eval(), Xe, med)).mean(0)
            g0 = expl[(0, 'IG', 0, 'a')]
            sanity = {'top10_overlap': len(set(np.argsort(-g0)[:10]) & set(np.argsort(-ig_r)[:10])) / 10,
                      'rho': spearmanr(g0, ig_r).correlation}
        expl[(s, 'LIME', LIME_N, 'a')] = np.abs(lime_local(fn, Xe, X, num_samples=LIME_N, seed=s)).mean(0)
    rec = {'pipeline': p, 'genes': genes, 'cv_c': cv_c, 'expl': expl, 'sanity': sanity, 'runtime_s': time.time() - t0}
    save(f'grid/{key}', rec)
    return rec


GRID_RES = []
for i, p in enumerate(PIPELINES):
    rec = run_pipeline(p)
    GRID_RES.append(rec)
    print(f"  [{i + 1:2d}/{len(PIPELINES)}] {pipe_key(p):30s} CV C = {np.mean(rec['cv_c']):.3f}  "
          f"IG sanity overlap = {rec['sanity']['top10_overlap']:.1f}  "
          f"IG top-3 = {', '.join(top_genes(rec['expl'][(0, 'IG', 0, 'a')], CLIN_COLS + rec['genes'], 3))}  "
          f"({rec['runtime_s']:.0f}s)")

D3 = pd.DataFrame([{**r['pipeline'], 'cv_C_mean': np.mean(r['cv_c']), 'cv_C_sd': np.std(r['cv_c']),
                    'IG_sanity_top10_overlap': r['sanity']['top10_overlap'], 'IG_sanity_rho': r['sanity']['rho'],
                    'IG_top5': ', '.join(top_genes(r['expl'][(0, 'IG', 0, 'a')], CLIN_COLS + r['genes'], 5))}
                   for r in GRID_RES])
print(D3.round(3).to_string(index=False))
D3.to_csv(f'{OUT}/D3_pipelines.csv', index=False)

In [ ]:
# ===== D4. Item 3 -- How much of the explanation disagreement does each choice cause? =====
# Every pair of explanations is compared on the genes both contain (1 - Spearman) and on their top-20 genes
# (1 - Jaccard). Each pair is coded by which choices differ between its two explanations. Two summaries:
#   one-factor-at-a-time: mean disagreement over pairs that differ in exactly ONE choice (most intuitive),
#   Shapley R^2: share of the explained variance of disagreement attributable to each choice (all pairs).
# CIs: cluster bootstrap over pipelines.
FACTORS = ['transform', 'selection', 'k', 'scaler', 'model seed', 'method family', 'SHAP budget', 'SHAP re-run']
ex_rows, vecs = [], []
for r in GRID_RES:
    names = CLIN_COLS + r['genes']
    for (s, fam, budget, run), v in r['expl'].items():
        ex_rows.append({**r['pipeline'], 'seed': s, 'family': fam, 'budget': budget, 'run': run,
                        'pipe': pipe_key(r['pipeline'])})
        vecs.append(pd.Series(v, index=names))
EX = pd.DataFrame(ex_rows)
universe = pd.Index(sorted(set().union(*[set(v.index) for v in vecs])))
V = np.full((len(vecs), len(universe)), np.nan, dtype=np.float32)
for i, v in enumerate(vecs):
    V[i, universe.get_indexer(v.index)] = v.values
TOP20 = [set(top_genes(v.values, list(v.index))) for v in vecs]


def pair_table():
    ck = load('D4_pairs')
    if ck is not None and len(ck) == len(vecs) * (len(vecs) - 1) // 2:
        return ck
    I, J = np.triu_indices(len(vecs), 1)
    rho = np.empty(len(I)); jac = np.empty(len(I))
    for n_, (i, j) in enumerate(zip(I, J)):
        m = ~np.isnan(V[i]) & ~np.isnan(V[j])
        rho[n_] = spearmanr(V[i, m], V[j, m]).correlation if m.sum() > 3 else np.nan
        jac[n_] = jaccard(TOP20[i], TOP20[j])
    a, b = EX.iloc[I].reset_index(drop=True), EX.iloc[J].reset_index(drop=True)
    P = pd.DataFrame({'i': I, 'j': J, 'd_rho': 1 - rho, 'd_jac': 1 - jac, 'pipe_i': a['pipe'].values, 'pipe_j': b['pipe'].values})
    for f, col in [('transform', 'transform'), ('selection', 'selection'), ('k', 'k'), ('scaler', 'scaler'),
                   ('model seed', 'seed'), ('method family', 'family')]:
        P[f] = (a[col].values != b[col].values).astype(int)
    both_shap = (a['family'].values == 'SHAP') & (b['family'].values == 'SHAP')
    P['SHAP budget'] = (both_shap & (a['budget'].values != b['budget'].values)).astype(int)
    P['SHAP re-run'] = (both_shap & (a['budget'].values == b['budget'].values) & (a['run'].values != b['run'].values)).astype(int)
    save('D4_pairs', P)
    return P


P = pair_table()
X_f = P[FACTORS].values.astype(float)
pipes = sorted(EX['pipe'].unique()); pidx = {p: i for i, p in enumerate(pipes)}
PI, PJ = P['pipe_i'].map(pidx).values, P['pipe_j'].map(pidx).values


def shapley_r2(y, w):
    """Exact Shapley (LMG) decomposition of the weighted R^2 over the factor indicators, via normal equations."""
    p = len(FACTORS); cache = {}
    A = np.column_stack([np.ones(len(y)), X_f])
    Gm = (A * w[:, None]).T @ A; bv = (A * w[:, None]).T @ y
    ybar = np.average(y, weights=w); sst = np.sum(w * (y - ybar) ** 2)
    def R(S):
        if S not in cache:
            idx = [0] + [c + 1 for c in S]
            beta = np.linalg.lstsq(Gm[np.ix_(idx, idx)], bv[idx], rcond=None)[0]
            sse = np.sum(w * y ** 2) - beta @ bv[idx]
            cache[S] = 1 - sse / sst if S else 0.0
        return cache[S]
    out = np.zeros(p)
    for f in range(p):
        others = [g for g in range(p) if g != f]
        for k_ in range(p):
            for S in combinations(others, k_):
                wgt = factorial(k_) * factorial(p - k_ - 1) / factorial(p)
                out[f] += wgt * (R(tuple(sorted(S + (f,)))) - R(S))
    return out, R(tuple(range(p)))


def ofat(y, w):
    single = X_f.sum(1) == 1
    return np.array([np.average(y[single & (X_f[:, f] == 1)], weights=w[single & (X_f[:, f] == 1)])
                     if (w[single & (X_f[:, f] == 1)] > 0).any() else np.nan for f in range(len(FACTORS))])


N_BOOT_D4 = 50 if QUICK_RUN else 300
rsb = np.random.RandomState(42)
D4 = {}
for metric, lab in [('d_rho', '1 - Spearman'), ('d_jac', '1 - top-20 Jaccard')]:
    ok = P[metric].notna().values
    y = P[metric].values[ok]; X_f_ok = X_f[ok]; w1 = np.ones(ok.sum())
    X_backup = X_f; X_f = X_f_ok
    sh, r2_full = shapley_r2(y, w1)
    est = ofat(y, w1)
    boots_ofat, boots_sh = [], []
    for _ in range(N_BOOT_D4):
        cnt = np.bincount(rsb.randint(0, len(pipes), len(pipes)), minlength=len(pipes))
        w = cnt[PI[ok]] * cnt[PJ[ok]].astype(float)
        if w.sum() == 0:
            continue
        boots_ofat.append(ofat(y, w)); boots_sh.append(shapley_r2(y, w)[0])
    X_f = X_backup
    bo, bs = np.array(boots_ofat), np.array(boots_sh)
    D4[lab] = pd.DataFrame({'factor': FACTORS, 'one_factor_mean': est,
                            'ofat_CI_lo': np.nanpercentile(bo, 2.5, 0), 'ofat_CI_hi': np.nanpercentile(bo, 97.5, 0),
                            'n_single_factor_pairs': [int(((X_f_ok.sum(1) == 1) & (X_f_ok[:, f] == 1)).sum()) for f in range(len(FACTORS))],
                            'shapley_R2': sh, 'share_of_explained_%': 100 * sh / sh.sum(),
                            'share_CI_lo': np.nanpercentile(100 * bs / bs.sum(1, keepdims=True), 2.5, 0),
                            'share_CI_hi': np.nanpercentile(100 * bs / bs.sum(1, keepdims=True), 97.5, 0)})
    print(f"\n=== {lab}: total R^2 = {r2_full:.3f} over {ok.sum():,} explanation pairs ===")
    print(D4[lab].round(3).to_string(index=False))
    D4[lab].to_csv(f'{OUT}/D4_decomposition_{metric}.csv', index=False)
record('D4_decomposition', {k: v.to_dict('records') for k, v in D4.items()})

# Figure: one-factor-at-a-time disagreement (left) and share of explained variance (right)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
t = D4['1 - top-20 Jaccard'].sort_values('one_factor_mean')
axes[0].barh(t.factor, t.one_factor_mean, xerr=[t.one_factor_mean - t.ofat_CI_lo, t.ofat_CI_hi - t.one_factor_mean],
             color='#2a78d6', ecolor='#555555', capsize=3)
axes[0].set_xlabel('Top-20 gene disagreement (1 - Jaccard)\nwhen only this choice changes'); axes[0].set_xlim(0, 1)
t2 = D4['1 - Spearman'].set_index('factor').loc[t.factor]
axes[1].barh(t2.index, t2['share_of_explained_%'],
             xerr=[t2['share_of_explained_%'] - t2.share_CI_lo, t2.share_CI_hi - t2['share_of_explained_%']],
             color='#1baf7a', ecolor='#555555', capsize=3)
axes[1].set_xlabel('Share of explained disagreement (%)\n(Shapley decomposition of $R^2$, 1 - Spearman)')
for ax in axes:
    ax.grid(axis='x', alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_disagreement_decomposition.png', dpi=500, bbox_inches='tight'); plt.close()
print(f"Saved: {OUT}/fig_disagreement_decomposition.png")

In [ ]:
# ===== D5. Item 4 -- Survival-specific explanations at 6, 12 and 24 months =====
# (a) time-dependent IG on S(t|x); (b) SurvSHAP(t) = KernelSHAP applied to S(t|x) at each t (Krzyzinski et al. 2023);
# (c) SurvLIME (Kovalev et al. 2020) implemented from its objective. For a proportional-hazards model such as ours,
#     log H(t|x) = log H0(t) + f(x), so SurvLIME's time terms cancel and it reduces to LIME on the log-risk; we
#     verify this numerically. (d) Contrast: a random survival forest, which is not proportional-hazards.
BRES = breslow(r_all, T, E_int)
H0_T = torch.tensor(H0_at(BRES, TIMES), dtype=torch.float32)
C_T = BRES[2]


class SurvOut(nn.Module):
    def forward(self, r):
        return torch.exp(-H0_T[None, :] * torch.exp(r[:, None] - C_T))


F_surv = Flat(model_full, out=SurvOut()).eval()
fn_surv = np_fn(F_surv)
SURV_N = 15 if QUICK_RUN else 60
SURV_IDX = np.sort(np.random.RandomState(7).choice(N, SURV_N, replace=False))
SB = 256 if QUICK_RUN else 2048

ck = load('D5_network')
if ck is None:
    t0 = time.time()
    tig = {t: np.abs(ig_attr(F_surv, X_all, MED_ALL, target=j)).mean(0) for j, t in enumerate(TIMES)}
    sv_t = kernel_shap(fn_surv, X_all[SURV_IDX], X_all, SB, seed=42, l1=False)          # (n, p, 3)
    sv_r = kernel_shap(pred_full, X_all[SURV_IDX], X_all, SB, seed=42, l1=False)        # risk, same patients
    lime_r = lime_local(pred_full, X_all[SURV_IDX], X_all, num_samples=LIME_N if QUICK_RUN else 1000, seed=42)
    ck = {'tig': tig, 'survshap': np.abs(sv_t).mean(0), 'shap_risk': np.abs(sv_r).mean(0),
          'lime_risk': np.abs(lime_r).mean(0), 'runtime_s': time.time() - t0}
    save('D5_network', ck)


def survlime(x, fn_logH, times, Xref, n_samples=1000, ridge=1.0, seed=0, H0_ref=None):
    """SurvLIME for one patient: weighted least squares between the black box's log cumulative hazard and a
    local Cox model log H0(t) + b.z over the time grid, with a per-time intercept and time weights v(t)."""
    rs = np.random.RandomState(seed)
    sd = Xref.std(0) + 1e-8
    Z = x + rs.normal(0, 1, (n_samples, len(x))) * sd
    d = np.sqrt((((Z - x) / sd) ** 2).sum(1)); width = 0.75 * np.sqrt(len(x))
    w = np.exp(-(d ** 2) / width ** 2)
    LH = fn_logH(Z.astype(f32))                                   # (n_samples, n_times)
    if H0_ref is not None:
        LH = LH - np.log(H0_ref)[None, :]
    v = np.ones(len(times)) / len(times)
    zbar = np.average(Z, axis=0, weights=w); Zc = Z - zbar
    A = (Zc * w[:, None]).T @ Zc * v.sum() + ridge * np.eye(Z.shape[1])
    rhs = sum(v[j] * (Zc * w[:, None]).T @ (LH[:, j] - np.average(LH[:, j], weights=w)) for j in range(len(times)))
    return np.linalg.solve(A, rhs)


def logH_net(Z):
    with torch.no_grad():
        r = F_full(torch.tensor(Z)).numpy()
    return np.log(H0_at(BRES, TIMES))[None, :] + (r - C_T)[:, None]


ck2 = load('D5_survlime')
if ck2 is None:
    t0 = time.time()
    sl = np.array([survlime(X_all[i], logH_net, TIMES, X_all, n_samples=300 if QUICK_RUN else 1000, seed=int(i))
                   for i in SURV_IDX])
    # equivalence check: the same regression on the log-risk alone (time-free LIME with the same samples/weights)
    lr = np.array([survlime(X_all[i], lambda Z: pred_full(Z)[:, None], [0.0], X_all,
                            n_samples=300 if QUICK_RUN else 1000, seed=int(i)) for i in SURV_IDX])
    ck2 = {'survlime': np.abs(sl).mean(0), 'max_abs_diff_vs_logrisk': float(np.abs(sl - lr).max()),
           'runtime_s': time.time() - t0}
    save('D5_survlime', ck2)
print(f"SurvLIME vs the same regression on log-risk only: max |coefficient difference| = "
      f"{ck2['max_abs_diff_vs_logrisk']:.2e} (proportional hazards -> identical)")

# (d) Random survival forest contrast (not proportional hazards)
ck3 = load('D5_rsf')
if ck3 is None:
    t0 = time.time()
    rsf = fit_rsf(X_all.astype(float), Surv.from_arrays(E.astype(bool), T), n_estimators=200)
    ut = getattr(rsf, 'unique_times_', getattr(rsf, 'event_times_', None))
    jt = np.clip(np.searchsorted(ut, TIMES, side='right') - 1, 0, None)
    def fn_rsf(Z):
        return rsf.predict_survival_function(np.asarray(Z, float), return_array=True)[:, jt]
    n_rsf = 10 if QUICK_RUN else 30
    sv_rsf = kernel_shap(fn_rsf, X_all[SURV_IDX[:n_rsf]], X_all, 128 if QUICK_RUN else 1024, seed=42, l1=False, n_bg=10)
    ck3 = {'survshap_rsf': np.abs(sv_rsf).mean(0), 'runtime_s': time.time() - t0}
    save('D5_rsf', ck3)

rows = []
for j, t in enumerate(TIMES):
    rows.append({'t (months)': t,
                 'tIG vs IG(risk) rho': spearmanr(ck['tig'][t], IG_GLOBAL).correlation,
                 'tIG top-20 Jaccard vs IG': jaccard(top_genes(ck['tig'][t], FEATURES), top_genes(IG_GLOBAL, FEATURES)),
                 'SurvSHAP(t) vs SHAP(risk) rho': spearmanr(ck['survshap'][:, j], ck['shap_risk']).correlation,
                 'SurvSHAP(t) vs tIG rho': spearmanr(ck['survshap'][:, j], ck['tig'][t]).correlation,
                 'RSF SurvSHAP(t) vs RSF t=6 rho': spearmanr(ck3['survshap_rsf'][:, j], ck3['survshap_rsf'][:, 0]).correlation,
                 'net SurvSHAP(t) vs net t=6 rho': spearmanr(ck['survshap'][:, j], ck['survshap'][:, 0]).correlation,
                 'tIG top-5': ', '.join(top_genes(ck['tig'][t], FEATURES, 5)),
                 'RSF SurvSHAP(t) top-5': ', '.join(top_genes(ck3['survshap_rsf'][:, j], FEATURES, 5))})
D5 = pd.DataFrame(rows)
print(D5.round(3).to_string(index=False))
print(f"SurvLIME vs LIME(risk, training-mean sampling) rho = {spearmanr(ck2['survlime'], ck['lime_risk']).correlation:.3f}; "
      f"SurvLIME vs IG rho = {spearmanr(ck2['survlime'], IG_GLOBAL).correlation:.3f}")
D5.to_csv(f'{OUT}/D5_time_dependent.csv', index=False)
record('D5_survival_explainers', {'table': D5.to_dict('records'),
                                  'survlime_equivalence_max_diff': ck2['max_abs_diff_vs_logrisk'],
                                  'survlime_vs_lime_rho': spearmanr(ck2['survlime'], ck['lime_risk']).correlation,
                                  'survlime_vs_ig_rho': spearmanr(ck2['survlime'], IG_GLOBAL).correlation})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, A, ttl in [(axes[0], ck['survshap'], 'Gated fusion (proportional hazards)'),
                   (axes[1], ck3['survshap_rsf'], 'Random survival forest')]:
    top = np.argsort(-A.mean(1))[:8]
    for j, t in enumerate(TIMES):
        ax.barh(np.arange(8) + (j - 1) * 0.27, A[top, j] / A[:, j].sum(), height=0.27, label=f'{int(t)} months',
                color=['#9ec9f0', '#2a78d6', '#0b3d75'][j])
    ax.set_yticks(np.arange(8)); ax.set_yticklabels([FEATURES[i] for i in top]); ax.invert_yaxis()
    ax.set_title(ttl, fontsize=10); ax.set_xlabel('Share of SurvSHAP(t) mass'); ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_survshap_time.png', dpi=500, bbox_inches='tight'); plt.close()
print(f"Saved: {OUT}/fig_survshap_time.png")

In [ ]:
# ===== D6. Item 4 -- Survival metrics with bootstrap CIs, calibration and D-calibration =====
OOF = load_v5('extended_eval_oof_preds')
if OOF is None:
    OOF, _ = run_cv(Xc_raw, Xg_clip, T, E, folds=FOLDS)
extra = load('D6_baselines')
if extra is None:
    extra, _ = run_cv(Xc_raw, Xg_clip, T, E, models=('Coxnet', 'RSF'), folds=FOLDS)
    save('D6_baselines', extra)
ALLM = {**{k: OOF[k] for k in ['Clinical MLP', 'Genomic MLP', 'Gated fusion', 'Concat fusion']}, **extra}
HZ = [6, 12, 24]; TAU = 36.0
N_BOOT_D6 = 30 if QUICK_RUN else 300


def fold_metrics(tr, te, r_tr, r_te, ii=None):
    ii = np.arange(len(te)) if ii is None else ii
    Tt, Et, rt = T[te][ii], E[te][ii], np.asarray(r_te)[ii]
    y_tr = Surv.from_arrays(E[tr].astype(bool), T[tr])
    y_te = Surv.from_arrays(Et.astype(bool), np.minimum(Tt, T[tr].max() - 1e-3))
    out = {'Harrell C': harrell_c(Tt, Et.astype(int), rt)}
    try:
        out['Uno C'] = concordance_index_ipcw(y_tr, y_te, rt, tau=TAU)[0]
    except Exception:
        out['Uno C'] = np.nan
    for h in HZ:
        try:
            out[f'AUC@{h}'] = cumulative_dynamic_auc(y_tr, y_te, rt, times=[h])[0][0]
        except Exception:
            out[f'AUC@{h}'] = np.nan
    try:
        times = np.linspace(max(np.percentile(Tt, 10), 1.0), min(np.percentile(Tt, 80), T[tr].max() - 0.01), 15)
        out['IBS'] = integrated_brier_score(y_tr, y_te, surv_from_risk(breslow(r_tr, T[tr], E_int[tr]), rt, times), times)
    except Exception:
        out['IBS'] = np.nan
    return out


ck = load('D6_metrics')
if ck is None:
    rsb = np.random.RandomState(42); ck = {}
    for m, pr in ALLM.items():
        point = [fold_metrics(tr, te, pr['tr'][f], pr['te'][f]) for f, (tr, te) in enumerate(FOLDS)]
        boot = []
        for _ in range(N_BOOT_D6):
            boot.append(pd.DataFrame([fold_metrics(tr, te, pr['tr'][f], pr['te'][f], rsb.randint(0, len(te), len(te)))
                                      for f, (tr, te) in enumerate(FOLDS)]).mean())
        ck[m] = {'point': pd.DataFrame(point).mean(), 'boot': pd.DataFrame(boot)}
    save('D6_metrics', ck)
rows = []
for m, v in ck.items():
    row = {'model': m}
    for c in v['point'].index:
        row[c] = f"{v['point'][c]:.3f} [{np.nanpercentile(v['boot'][c], 2.5):.3f}, {np.nanpercentile(v['boot'][c], 97.5):.3f}]"
    rows.append(row)


# Calibration on pooled out-of-fold predictions: grouped (quintiles) at 12 and 24 months + D-calibration
def oof_survival(pr, times):
    S = np.zeros((N, len(times))); S_own = np.zeros(N)
    for f, (tr, te) in enumerate(FOLDS):
        b = breslow(pr['tr'][f], T[tr], E_int[tr])
        S[te] = surv_from_risk(b, pr['te'][f], times)
        S_own[te] = np.exp(-np.exp(np.asarray(pr['te'][f]) - b[2]) * H0_at(b, T[te]))
    return S, S_own


def d_calibration(S_own, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); counts = np.zeros(n_bins)
    for s, e in zip(np.clip(S_own, 1e-9, 1), E_int):
        if e == 1:
            counts[min(int(s * n_bins), n_bins - 1)] += 1
        else:
            for b in range(n_bins):
                lo, hi = edges[b], edges[b + 1]
                if hi <= s:
                    counts[b] += (hi - lo) / s
                elif lo < s:
                    counts[b] += (s - lo) / s
    exp_ = N / n_bins
    return float(chi2.sf(((counts - exp_) ** 2 / exp_).sum(), n_bins - 1))


cal_rows, cal_pts = [], {}
for m, pr in ALLM.items():
    S, S_own = oof_survival(pr, [12, 24])
    row = {'model': m, 'D-calibration p': d_calibration(S_own)}
    for j, h in enumerate([12, 24]):
        q = pd.qcut(1 - S[:, j], 5, labels=False, duplicates='drop'); pts = []
        for g in np.unique(q):
            km = KaplanMeierFitter().fit(T[q == g], E[q == g])
            pts.append((float((1 - S[q == g, j]).mean()), float(1 - km.survival_function_at_times(h).values[0])))
        pts = np.array(pts); cal_pts[(m, h)] = pts
        row[f'grouped cal. error @{h}'] = float(np.mean(np.abs(pts[:, 0] - pts[:, 1])))
    cal_rows.append(row)
D6 = pd.DataFrame(rows).merge(pd.DataFrame(cal_rows).round(3), on='model')
print(D6.to_string(index=False))
print("Brackets: 95% bootstrap CI (patients resampled within each test fold). IBS lower = better. "
      "D-calibration p > 0.05 = no evidence of miscalibration (Haider et al. 2020).")
D6.to_csv(f'{OUT}/D6_survival_metrics.csv', index=False)
record('D6_survival_metrics', D6.to_dict('records'))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, h in zip(axes, [12, 24]):
    ax.plot([0, 1], [0, 1], color='#8a8a85', ls='--', lw=1)
    for m, c in [('Gated fusion', '#2a78d6'), ('Coxnet', '#eb6834'), ('Clinical MLP', '#1baf7a')]:
        pts = cal_pts[(m, h)]; ax.plot(pts[:, 0], pts[:, 1], marker='o', color=c, lw=2, label=m)
    ax.set_xlabel(f'Predicted risk of death by {h} months'); ax.set_ylabel('Observed (Kaplan-Meier)')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.spines[['top', 'right']].set_visible(False); ax.set_title(f'{h} months', fontsize=10)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_calibration.png', dpi=500, bbox_inches='tight'); plt.close()
print(f"Saved: {OUT}/fig_calibration.png")

In [ ]:
# ===== D7. Item 1 -- External pancreatic cancer cohorts (GEO), downloaded and harmonised =====
# Each cohort is parsed automatically: probe -> gene symbol from the platform table, tumour samples only,
# survival time and status from the sample characteristics. CHECK THE PRINTED SUMMARY: if a field is picked up
# wrongly, set it in OVERRIDES (column names are printed) and re-run this cell.
import GEOparse

GEO_DIR = f'{DATA_DIR}/geo'; os.makedirs(GEO_DIR, exist_ok=True)
EXTERNAL = ['GSE62452', 'GSE28735', 'GSE71729', 'GSE57495']
OVERRIDES = {   # e.g. 'GSE62452': {'time': 'characteristics_ch1.3.survival months', 'event': '...', 'event_is_censor': False}
}
EXCLUDE_TISSUE = r'normal|non-?\s?tumou?r|adjacent|metasta|cell line|pancreatitis|liver|lymph node|lung'


def _num(s):
    return pd.to_numeric(pd.Series(s).astype(str).str.extract(r'(-?\d+\.?\d*)')[0], errors='coerce').values


def parse_event(v):
    s = str(v).strip().lower()
    if s in ('1', '1.0', 'dead', 'deceased', 'died', 'yes', 'true', 'event', 'death', 'dod', 'dead of disease'):
        return 1.0
    if s in ('0', '0.0', 'alive', 'living', 'no', 'false', 'censored', 'awd', 'ned', 'alive with disease'):
        return 0.0
    if 'dead' in s or 'deceas' in s:
        return 1.0
    if 'alive' in s or 'living' in s:
        return 0.0
    return np.nan


def parse_stage(v):
    s = str(v).upper()
    m = re.search(r'(IV|III|II|I)[ABC]?\b', s) or re.search(r'\b([1-4])[ABC]?\b', s)
    if not m:
        return np.nan
    return {'I': 1, 'II': 2, 'III': 3, 'IV': 4, '1': 1, '2': 2, '3': 3, '4': 4}[m.group(1)]


def probe_symbols(gpl_table, ids):
    tab = gpl_table.copy(); tab['ID'] = tab['ID'].astype(str)
    cols = {c.lower().replace('_', ' ').strip(): c for c in tab.columns}
    sym = None
    for key in ['gene symbol', 'symbol', 'genesymbol', 'gene name', 'orf', 'gene']:
        if key in cols:
            sym = tab[cols[key]].astype(str).str.split(r'\s*///\s*').str[0].str.strip(); break
    if sym is None and 'gene assignment' in cols:
        sym = tab[cols['gene assignment']].astype(str).str.split('//').str[1].str.strip()
    if sym is None:
        return None
    mp = dict(zip(tab['ID'], sym))
    return pd.Series([mp.get(str(i)) for i in ids], index=ids)


def load_geo_cohort(gse_id):
    ck = load(f'ext_{gse_id}')
    if ck is not None:
        return ck
    gse = GEOparse.get_GEO(geo=gse_id, destdir=GEO_DIR, silent=True)
    ph = gse.phenotype_data.copy()
    ph.columns = [str(c) for c in ph.columns]
    low = {c: c.lower() for c in ph.columns}
    ov = OVERRIDES.get(gse_id, {})
    # tumour samples only
    tissue_cols = [c for c in ph.columns if re.search(r'tissue|sample type|source|type|histolog|title|disease', low[c])]
    excl = np.zeros(len(ph), bool)
    for c in tissue_cols:
        excl |= ph[c].astype(str).str.lower().str.contains(EXCLUDE_TISSUE, regex=True).values
    # survival time / status
    tcol = ov.get('time') or next((c for c in ph.columns if re.search(r'characteristics', low[c]) and
                                   re.search(r'(surv|os\b|overall|follow|time).*(month|mo\b|day|time|year)|month', low[c])
                                   and not re.search(r'status|event|censor|vital', low[c])), None)
    ecol = ov.get('event') or next((c for c in ph.columns if re.search(r'characteristics', low[c]) and
                                    re.search(r'status|event|death|dead|vital|censor', low[c])), None)
    if tcol is None or ecol is None:
        print(f"  {gse_id}: survival fields not found. Characteristic columns: "
              f"{[c for c in ph.columns if 'characteristics' in low[c]]}")
        return None
    tt = _num(ph[tcol])
    if 'day' in low[tcol]:
        tt = tt / 30.44
    elif 'year' in low[tcol]:
        tt = tt * 12
    ee = np.array([parse_event(v) for v in ph[ecol]])
    if ov.get('event_is_censor', 'censor' in low[ecol] and not re.search(r'death|dead|event|status|vital', low[ecol])):
        ee = 1 - ee
    age_col = next((c for c in ph.columns if re.search(r'characteristics.*\bage\b', low[c])), None)
    st_col = next((c for c in ph.columns if re.search(r'characteristics.*(stage|ajcc|tnm)', low[c])), None)
    keep = ~excl & ~np.isnan(tt) & ~np.isnan(ee) & (tt > 0)
    gsm = ph.index[keep]
    # expression: probes x samples -> genes x samples (highest-mean probe per gene)
    gpl_id = list(gse.gpls)[0]
    ex = gse.pivot_samples('VALUE')[list(gsm)].apply(pd.to_numeric, errors='coerce')
    sym = probe_symbols(gse.gpls[gpl_id].table, ex.index)
    if (sym is None or sym.isin(gene_cols).sum() < 100) and pd.Index(ex.index.astype(str)).isin(gene_cols).sum() > 100:
        sym = pd.Series(ex.index.astype(str), index=ex.index)       # IDs are already gene symbols
    if sym is None:
        print(f"  {gse_id}: no gene-symbol column in {gpl_id}; columns = {list(gse.gpls[gpl_id].table.columns)}")
        return None
    ex = ex[sym.notna().values & ~sym.isin(['', 'nan', '---', 'None']).values]
    ex['__sym'] = sym.loc[ex.index].values
    ex = ex.dropna(subset=list(gsm), how='all')
    ex['__mean'] = ex[list(gsm)].mean(1)
    ex = ex.sort_values('__mean', ascending=False).drop_duplicates('__sym').set_index('__sym').drop(columns='__mean')
    ex = ex.T.astype(f32)
    if np.nanmax(ex.values) > 100:
        ex = np.log2(ex.clip(lower=0) + 1)
    ex = ex.fillna(ex.median())
    out = {'id': gse_id, 'platform': gpl_id, 'expr': ex, 'T': tt[keep].astype(f32), 'E': ee[keep].astype(f32),
           'age': _num(ph.loc[gsm, age_col]) if age_col else np.full(len(gsm), np.nan),
           'stage': np.array([parse_stage(v) for v in ph.loc[gsm, st_col]]) if st_col else np.full(len(gsm), np.nan),
           'fields': {'time': tcol, 'event': ecol, 'age': age_col, 'stage': st_col},
           'n_excluded_non_tumour': int(excl.sum())}
    save(f'ext_{gse_id}', out)
    return out


EXT = {}
for gid in EXTERNAL:
    try:
        c = load_geo_cohort(gid)
    except Exception as e:
        print(f"  {gid}: failed to load ({type(e).__name__}: {e})"); c = None
    if c is None:
        continue
    EXT[gid] = c
    ov_genes = len(set(c['expr'].columns) & set(gene_cols))
    print(f"{gid} [{c['platform']}]: n = {len(c['T'])} tumours, deaths = {int(c['E'].sum())} "
          f"({c['E'].mean():.0%}), median time = {np.median(c['T']):.1f} mo, genes = {c['expr'].shape[1]:,} "
          f"(of the paper's 1,000: {ov_genes}); age {'yes' if np.isfinite(c['age']).any() else 'no'}, "
          f"stage {'yes' if np.isfinite(c['stage']).any() else 'no'}; non-tumour excluded = {c['n_excluded_non_tumour']}")
    print(f"     fields used: {c['fields']}")
print("\nSanity: PDAC cohorts usually have 60-90% deaths and a median time of 12-25 months. If a cohort looks "
      "inverted (e.g. 20% deaths) the event field is probably a censoring flag: set OVERRIDES and re-run.")

In [ ]:
# ===== D8. Item 1 -- External validation of predictions and explanations =====
HARMONISE = ['cohort-wise robust scaling', 'TCGA scaler (none)', 'quantile-matched to TCGA']
EXT_SB, EXT_SN = (128, 10) if QUICK_RUN else (2048, 60)


def quantile_match(A, ref):
    out = np.empty_like(A)
    for j in range(A.shape[1]):
        q = (rankdata(A[:, j]) - 0.5) / len(A)
        out[:, j] = np.quantile(ref[:, j], q)
    return out


def external_analysis(gid, c):
    ck = load(f'D8_{gid}')
    if ck is not None:
        return ck
    t0 = time.time()
    G = [g for g in gene_cols if g in c['expr'].columns]
    gi = [gene_cols.index(g) for g in G]
    names = CLIN_COLS + G
    # TCGA model on the shared genes
    model, sc, sg, X_tcga = fit_full(Xc_raw, Xg_clip[:, gi], T, E)
    F = Flat(model).eval(); fn = np_fn(F)
    oof, _ = run_cv(Xc_raw, Xg_clip[:, gi], T, E, models=('Gated fusion',), folds=FOLDS)
    tcga_cv = float(np.mean([harrell_c(T[te], E_int[te], oof['Gated fusion']['te'][f]) for f, (tr, te) in enumerate(FOLDS)]))
    ig_tcga = np.abs(ig_attr(F, X_tcga, np.median(X_tcga, 0))).mean(0)
    # external clinical: TCGA scaler (same units); missing values -> TCGA median
    med_c = np.median(Xc_raw, 0)
    xc = np.column_stack([np.where(np.isfinite(c['age']), c['age'], med_c[0]),
                          np.where(np.isfinite(c['stage']), c['stage'], med_c[1])]).astype(f32)
    Xc_e = sc.transform(xc).astype(f32)
    A = clip_pct(c['expr'][G].values.astype(f32))
    Te, Ee = c['T'], c['E']
    res = {'n': len(Te), 'deaths': int(Ee.sum()), 'n_genes': len(G), 'tcga_cv_C_shared_genes': tcga_cv,
           'clinical_available': {'age': bool(np.isfinite(c['age']).any()), 'stage': bool(np.isfinite(c['stage']).any())},
           'harmonisation': {}}
    rsb = np.random.RandomState(42)
    for h in HARMONISE:
        if h.startswith('cohort'):
            Xg_e = RobustScaler().fit_transform(A)
        elif h.startswith('TCGA'):
            Xg_e = sg.transform(A)
        else:
            Xg_e = sg.transform(quantile_match(A, Xg_clip[:, gi]))
        X_e = np.hstack([Xc_e, Xg_e]).astype(f32)
        r = fn(X_e)
        boots = [harrell_c(Te[ii], Ee[ii].astype(int), r[ii]) for ii in (rsb.randint(0, len(Te), len(Te)) for _ in range(1000))]
        hi = r >= np.median(r)
        lr = logrank_test(Te[hi], Te[~hi], Ee[hi], Ee[~hi]).p_value
        ig_e = np.abs(ig_attr(F, X_e, np.median(X_e, 0))).mean(0)
        res['harmonisation'][h] = {'C': harrell_c(Te, Ee.astype(int), r), 'C_CI': (np.percentile(boots, 2.5), np.percentile(boots, 97.5)),
                                   'logrank_p': lr, 'ig_rho_vs_tcga': spearmanr(ig_e, ig_tcga).correlation,
                                   'ig_top20_jaccard_vs_tcga': jaccard(top_genes(ig_e, names), top_genes(ig_tcga, names))}
        if h.startswith('cohort'):
            X_main, ig_main = X_e, ig_e
    # finding 2 replicated externally: KernelSHAP budget vs IG on the external patients
    sub = np.sort(rsb.choice(len(Te), min(EXT_SN, len(Te)), replace=False))
    ig_sub = np.abs(ig_attr(F, X_main[sub], np.median(X_main, 0))).mean(0)
    sh_lo = np.abs(kernel_shap(fn, X_main[sub], X_main, SHAP_LO, seed=1)).mean(0)
    sh_lo2 = np.abs(kernel_shap(fn, X_main[sub], X_main, SHAP_LO, seed=2)).mean(0)
    sh_hi = np.abs(kernel_shap(fn, X_main[sub], X_main, EXT_SB, seed=3, l1=False)).mean(0)
    res['shap_budget'] = {'SHAP-low re-run rho': spearmanr(sh_lo, sh_lo2).correlation,
                          'SHAP-low vs IG rho': spearmanr(sh_lo, ig_sub).correlation,
                          'SHAP-high vs IG rho': spearmanr(sh_hi, ig_sub).correlation}
    # cohort-native model: are the same genes important when the model is learned from this cohort alone?
    if len(Te) >= 50:
        Xg_n = RobustScaler().fit_transform(A).astype(f32)
        torch.manual_seed(42)
        mn = train_model(GatedFusion(NC, len(G)), Xc_e, Xg_n, Te, Ee)
        Fn = Flat(mn).eval(); Xn = np.hstack([Xc_e, Xg_n]).astype(f32)
        ig_n = np.abs(ig_attr(Fn, Xn, np.median(Xn, 0))).mean(0)
        ig_nr = np.abs(ig_attr(Flat(randomise(mn)).eval(), Xn, np.median(Xn, 0))).mean(0)
        j_obs = jaccard(top_genes(ig_n, names), top_genes(ig_tcga, names))
        null = [jaccard(rsb.choice(G, 20, replace=False), top_genes(ig_tcga, names)) for _ in range(2000)]
        foldsn = list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xc_e, Ee))
        oofn, _ = run_cv(xc, A, Te, Ee, models=('Gated fusion',), folds=foldsn)
        res['native'] = {'cv_C': float(np.mean([harrell_c(Te[te], Ee[te].astype(int), oofn['Gated fusion']['te'][f])
                                                for f, (tr, te) in enumerate(foldsn)])),
                         'ig_rho_vs_tcga_model': spearmanr(ig_n, ig_tcga).correlation,
                         'top20_jaccard_vs_tcga_model': j_obs, 'top20_jaccard_null_mean': float(np.mean(null)),
                         'top20_jaccard_perm_p': float((np.sum(np.array(null) >= j_obs) + 1) / (len(null) + 1)),
                         'sanity_top10_overlap': len(set(np.argsort(-ig_n)[:10]) & set(np.argsort(-ig_nr)[:10])) / 10,
                         'top5': top_genes(ig_n, names, 5)}
    # gene-level replication: do the TCGA top-20 genes carry the same univariate survival direction here?
    tg = top_genes(ig_tcga, names)
    z_t = cox_score_z(Xg_clip[:, [gene_cols.index(g) for g in tg]], T, E)
    z_e = cox_score_z(A[:, [G.index(g) for g in tg]], Te, Ee)
    agree = int((np.sign(z_t) == np.sign(z_e)).sum())
    res['gene_replication'] = {'genes': tg, 'z_tcga': z_t.tolist(), 'z_ext': z_e.tolist(), 'sign_agreement': agree,
                               'sign_test_p': binomtest(agree, len(tg), 0.5, alternative='greater').pvalue,
                               'n_sig_ext_same_sign': int(((np.abs(z_e) > 1.96) & (np.sign(z_t) == np.sign(z_e))).sum())}
    res['tcga_top5'] = top_genes(ig_tcga, names, 5)
    res['runtime_s'] = time.time() - t0
    save(f'D8_{gid}', res)
    return res


D8 = {gid: external_analysis(gid, c) for gid, c in EXT.items()}
rows = []
for gid, r in D8.items():
    for h, v in r['harmonisation'].items():
        rows.append({'cohort': gid, 'n': r['n'], 'genes': r['n_genes'], 'harmonisation': h,
                     'external C': f"{v['C']:.3f} [{v['C_CI'][0]:.3f}, {v['C_CI'][1]:.3f}]", 'log-rank p': v['logrank_p'],
                     'IG rho vs TCGA': v['ig_rho_vs_tcga'], 'IG top-20 Jaccard vs TCGA': v['ig_top20_jaccard_vs_tcga']})
D8a = pd.DataFrame(rows)
print("Transported TCGA model (trained on TCGA only, applied unchanged):")
print(D8a.round(3).to_string(index=False))
rows = []
for gid, r in D8.items():
    row = {'cohort': gid, 'TCGA CV C (shared genes)': r['tcga_cv_C_shared_genes'], **r['shap_budget'],
           'TCGA top-20 genes, same survival direction': f"{r['gene_replication']['sign_agreement']}/20 "
                                                         f"(p = {r['gene_replication']['sign_test_p']:.3g})",
           'same direction and |z| > 1.96': r['gene_replication']['n_sig_ext_same_sign']}
    if 'native' in r:
        row.update({'native CV C': r['native']['cv_C'], 'native vs TCGA-model IG rho': r['native']['ig_rho_vs_tcga_model'],
                    'native top-20 Jaccard (null)': f"{r['native']['top20_jaccard_vs_tcga_model']:.3f} "
                                                    f"({r['native']['top20_jaccard_null_mean']:.3f}, p = {r['native']['top20_jaccard_perm_p']:.3g})",
                    'native sanity overlap': r['native']['sanity_top10_overlap']})
    rows.append(row)
D8b = pd.DataFrame(rows)
print("\nExplanation-level replication:")
print(D8b.round(3).to_string(index=False))
D8a.to_csv(f'{OUT}/D8_external_prediction.csv', index=False); D8b.to_csv(f'{OUT}/D8_external_explanations.csv', index=False)
record('D8_external', {gid: {k: v for k, v in r.items() if k != 'runtime_s'} for gid, r in D8.items()})

if D8:
    fig, ax = plt.subplots(figsize=(6.5, 0.6 + 0.55 * len(D8a)))
    labs = [f"{r.cohort} | {r.harmonisation}" for r in D8a.itertuples()]
    Cs = [D8[r.cohort]['harmonisation'][r.harmonisation]['C'] for r in D8a.itertuples()]
    los = [D8[r.cohort]['harmonisation'][r.harmonisation]['C_CI'][0] for r in D8a.itertuples()]
    his = [D8[r.cohort]['harmonisation'][r.harmonisation]['C_CI'][1] for r in D8a.itertuples()]
    yy = np.arange(len(labs))[::-1]
    ax.errorbar(Cs, yy, xerr=[np.array(Cs) - los, np.array(his) - Cs], fmt='o', color='#2a78d6', ecolor='#555555', capsize=3)
    ax.axvline(0.5, color='#8a8a85', ls='--', lw=1)
    ax.set_yticks(yy); ax.set_yticklabels(labs, fontsize=8); ax.set_xlabel("External Harrell's C (95% bootstrap CI)")
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.savefig(f'{OUT}/fig_external_forest.png', dpi=500, bbox_inches='tight'); plt.close()
    print(f"Saved: {OUT}/fig_external_forest.png")

In [ ]:
# ===== D9. Summary =====
with open(f'{OUT}/partD_results.json', 'w') as fh:
    json.dump(RESULTS, fh, indent=2, default=lambda o: float(o) if np.isscalar(o) else str(o))
print(f"Saved {OUT}/partD_results.json")
print("Send back: partD_results.json, all D*_*.csv files and the four fig_*.png files in", OUT)